[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/04_neural_networks/04_neural_networks.ipynb)

# 04. Neural Networks — XOR, Backpropagation, ReLU, Dropout, MNIST

> 원본 강의: [Lec 7–10, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — ML 실용 팁, 딥러닝 기본(XOR), Backpropagation, ReLU/초기화/Dropout

## 이 장을 배우는 이유

03번에서 만든 [로지스틱 회귀](../../../glossary.md#logistic-regression)는 꽤 쓸 만해 보였습니다. 그런데 이 모델에는
**아무리 오래 학습시켜도 절대 풀 수 없는 문제**가 있습니다.

입력이 두 개일 때, **둘 중 하나만 1이면 정답이 1**인 문제를 생각해봅시다.

| 입력 A | 입력 B | 정답 |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | **1** |
| 1 | 0 | **1** |
| 1 | 1 | 0 |

[XOR](../../../glossary.md#xor-problem)이라고 부르는 이 네 줄짜리 문제 앞에서 로지스틱 회귀는 무너집니다.
그리고 이건 사소한 예외가 아니었습니다. **1969년 이 사실이 알려지면서 신경망 연구는
십수 년간 얼어붙었습니다.** 이번 장은 무엇이 막혔고 무엇이 그것을 풀었는지에 대한 이야기입니다.

이번 장에서 배우는 것

- 로지스틱 회귀가 XOR을 못 푸는 이유 (그림 한 장이면 이해됩니다)
- [은닉층](../../../glossary.md#hidden-layer) 하나가 그것을 어떻게 해결하는지
- 층이 여러 개일 때 기울기를 구하는 방법 — [역전파](../../../glossary.md#backpropagation)
- 깊은 신경망에서 [시그모이드](../../../glossary.md#sigmoid) 대신 [ReLU](../../../glossary.md#relu)를 쓰는 이유
- [과적합](../../../glossary.md#overfitting)을 줄이는 [드롭아웃](../../../glossary.md#dropout)
- 손글씨 숫자 7만 장(MNIST)으로 실제 학습해보기

**소요 시간**: 50~60분. MNIST 학습 셀은 CPU에서 2~5분 걸립니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력과 그래프가 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**를 적어두었습니다.
- **학습이 포함된 셀은 수십 초~몇 분 걸립니다.** Colab이라면 `런타임 > 런타임 유형 변경`에서
  GPU를 켜면 훨씬 빨라집니다.
- 02·03번의 [경사 하강법](../../../glossary.md#gradient-descent)을 알고 있다고 가정합니다.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.


## 0. 준비 — 여기서부터 PyTorch를 씁니다

02·03번에서는 기울기를 손으로 계산해서 numpy로 적었습니다. `dW = np.mean((h - y) * x)` 처럼요.

**층이 여러 개가 되면 이걸 손으로 못 합니다.** 층이 3개면 미분을 3번 연쇄로 해야 하고,
층을 하나 추가할 때마다 식을 처음부터 다시 유도해야 합니다.

[PyTorch](../../../glossary.md#pytorch)는 그 계산을 대신 해줍니다. **우리가 할 일은 "예측을 어떻게 만드는지"만
적는 것이고, 기울기는 `loss.backward()` 한 줄이 알아서 구해줍니다.** 이것이 지금 PyTorch로
넘어가는 이유입니다.

Colab에서 GPU를 쓰려면 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU**로 설정하세요.


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q torch torchvision matplotlib koreanize-matplotlib


`device`는 **계산을 어디서 할지**를 담아두는 값입니다. GPU가 있으면 `cuda`, 없으면 `cpu`가 잡히고,
이후 모델과 데이터를 `.to(device)`로 그 장치에 올립니다. CPU여도 이 노트북은 다 돌아갑니다.

`torch.manual_seed(0)`은 난수 시드입니다. 신경망은 가중치를 무작위로 초기화하기 때문에,
시드를 고정하지 않으면 실행할 때마다 결과가 달라집니다.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False

# GPU(cuda)가 있으면 GPU를, 없으면 CPU를 쓴다
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)   # 가중치 초기값을 고정해 결과를 재현 가능하게 만든다
print("device:", device)


## 1. 문제 — 왜 직선 하나로는 XOR을 못 풀까

말로 설명하는 것보다 그려보는 게 빠릅니다. 입력 A를 가로축, 입력 B를 세로축에 놓고
네 점을 찍어봅시다. 정답이 1인 점은 빨간색, 0인 점은 파란색입니다.


In [ ]:
xor_inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]
xor_labels = [0, 1, 1, 0]

for (a, b), label in zip(xor_inputs, xor_labels):
    plt.scatter(a, b, s=300, c="red" if label else "blue",
                marker="o" if label else "s")
    plt.annotate(f"정답 {label}", (a, b), textcoords="offset points", xytext=(12, 8))

plt.xlim(-0.4, 1.6)
plt.ylim(-0.4, 1.4)
plt.xlabel("입력 A")
plt.ylabel("입력 B")
plt.title("XOR — 빨간 점(정답 1)과 파란 점(정답 0)을 직선 하나로 나눌 수 있을까?")
plt.grid(alpha=0.3)
plt.show()


**결과 읽는 법** — 빨간 점 두 개가 **대각선으로 마주 보고** 있습니다. 파란 점도 마찬가지입니다.

여기에 직선을 한 개만 그어서 빨강과 파랑을 갈라보세요. 어떻게 그어도 안 됩니다.
가로로 그으면 빨강 하나와 파랑 하나가 같은 쪽에 남고, 세로나 대각선도 마찬가지입니다.

03번에서 만든 로지스틱 회귀가 하는 일이 정확히 **"직선 하나 긋기"** 였습니다.
시그모이드는 직선의 결과를 확률로 바꿔줄 뿐, 경계 자체는 여전히 직선입니다.
그래서 이 문제는 **모델의 능력 밖**입니다. 학습을 더 오래 시켜서 될 일이 아닙니다.

이렇게 직선 하나로 나눌 수 없는 데이터를 "선형 분리가 불가능하다"고 하고,
그 경계선을 [결정 경계](../../../glossary.md#decision-boundary)라고 부릅니다.

정말 못 푸는지 직접 학습시켜서 확인해봅시다.


## 2. PyTorch로 모델을 만들고 학습시키기

먼저 데이터와 학습 함수를 만듭니다. PyTorch 코드가 처음이라면 낯선 이름이 많이 나오는데,
**02번에서 손으로 짠 루프와 하는 일이 정확히 같습니다.** 대응은 이렇습니다.

| 02번 (numpy) | PyTorch | 하는 일 |
|---|---|---|
| `y_pred = W * x + b` | `pred = model(X)` | 예측 |
| `cost = np.mean(...)` | `loss = loss_fn(pred, Y)` | 얼마나 틀렸나 |
| `dW = (2/m) * np.sum(...)` | `loss.backward()` | **기울기 계산 (자동)** |
| `W -= lr * dW` | `optimizer.step()` | 한 걸음 이동 |

새로 나오는 것은 `optimizer.zero_grad()` 하나입니다. PyTorch는 기울기를 **누적**하기 때문에,
매 걸음 시작할 때 이전 기울기를 지워줘야 합니다. **이 줄을 빼먹으면 학습이 이상해집니다.**


In [ ]:
# tensor: NumPy 배열과 비슷하지만 GPU 연산과 자동 미분이 되는 자료형
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], device=device)
Y = torch.tensor([[0.], [1.], [1.], [0.]], device=device)   # XOR 정답


def train(model, epochs=2000, lr=0.5):
    # SGD: 기울기 반대 방향으로 lr만큼 움직이는, 02번에서 손으로 한 그 방식
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()   # BCELoss: 03번에서 배운 이진 교차 엔트로피
    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()      # 지난 걸음의 기울기를 지운다
        pred = model(X)            # 1) 예측
        loss = loss_fn(pred, Y)    # 2) 얼마나 틀렸나
        loss.backward()            # 3) 기울기 계산 — 손으로 미분할 필요가 없다
        optimizer.step()           # 4) 한 걸음 이동
        losses.append(loss.item())
    return losses


print("학습 함수 준비 완료")


이제 **은닉층이 없는** 모델, 즉 03번의 로지스틱 회귀와 똑같은 구조를 만들어 XOR에 붙여봅니다.

- `nn.Linear(2, 1)`: 입력 2개를 받아 출력 1개를 만드는 층. `Wx + b`를 하는 부분입니다.
- `nn.Sigmoid()`: 그 결과를 0~1로 눌러 확률로 만듭니다.
- `nn.Sequential(...)`: 층을 순서대로 이어 붙입니다.

**실패하는 것이 정상입니다.** 무엇이 어떻게 실패하는지 보는 게 목적입니다.


In [ ]:
single_layer = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid()).to(device)
losses_single = train(single_layer)

with torch.no_grad():   # no_grad(): 기울기 추적을 끈다. 학습이 아니라 예측만 할 때 쓴다
    pred = single_layer(X)

print("입력 -> 예측값 (정답)")
for (a, b), p, t in zip(xor_inputs, pred.cpu().numpy().ravel(), xor_labels):
    print(f"  ({a}, {b}) -> {p:.3f}   (정답 {t})")
print(f"\n최종 loss: {losses_single[-1]:.4f}")


**결과 읽는 법** — 네 예측값이 모두 **0.5 근처**입니다. "모르겠다, 반반이다"라는 뜻입니다.

최종 loss도 **0.69 근처**에서 멈춥니다. 03번에서 본 그 숫자입니다.
`-log(0.5) = 0.693`, 즉 **아무것도 못 맞히고 전부 반반으로 찍을 때의 값**입니다.
2000번을 학습했는데도 학습 전과 다를 게 없습니다.

이것이 "모델의 능력 밖"이라는 말의 실제 모습입니다. 데이터를 더 주거나 더 오래 돌려도
바뀌지 않습니다. **모델의 구조를 바꿔야 합니다.**


## 3. 해결 — 직선 하나로 안 되면 여러 개 긋고 조합한다

아이디어는 단순합니다. **직선 하나로 못 나누면 여러 개 그으면 됩니다.**

XOR을 다시 보면 이렇게 풀 수 있습니다.

1. 직선 하나로 "A나 B 중 최소 하나는 1인가?"를 판단합니다
2. 다른 직선으로 "A와 B가 둘 다 1인가?"를 판단합니다
3. 두 판단을 조합합니다 — **1번은 맞고 2번은 아닌 경우**가 정답 1입니다

여기서 1번과 2번을 담당하는 것이 [은닉층](../../../glossary.md#hidden-layer)입니다.
"은닉(hidden)"이라 부르는 이유는 입력도 출력도 아니라서, 우리가 직접 정답을 알려주지 않는
**중간 계산 결과**이기 때문입니다. 무엇을 판단할지는 사람이 정해주지 않고 학습이 알아서 찾아냅니다.

코드에서 바뀌는 것은 층 두 줄뿐입니다. **학습 함수는 그대로 씁니다.**

```python
nn.Linear(2, 8),   # 입력 2개 -> 중간 판단 8개  (은닉층)
nn.Sigmoid(),
nn.Linear(8, 1),   # 중간 판단 8개 -> 최종 답 1개
nn.Sigmoid(),
```


In [ ]:
mlp = nn.Sequential(
    nn.Linear(2, 8),    # 은닉층: 중간 판단을 8개 만든다
    nn.Sigmoid(),
    nn.Linear(8, 1),    # 출력층: 8개의 판단을 모아 최종 답 1개로
    nn.Sigmoid(),
).to(device)
losses_mlp = train(mlp)

with torch.no_grad():
    pred = mlp(X)

print("입력 -> 예측값 (정답)")
for (a, b), p, t in zip(xor_inputs, pred.cpu().numpy().ravel(), xor_labels):
    print(f"  ({a}, {b}) -> {p:.3f}   (정답 {t})")
print(f"\n최종 loss: {losses_mlp[-1]:.4f}")


**결과 읽는 법** — 이번에는 예측값이 **0과 1 쪽으로 확실히 갈립니다.** loss도 0.69보다 훨씬 아래로
내려갑니다. 층 두 줄을 넣었을 뿐인데 못 풀던 문제가 풀렸습니다.

**이럴 때는 이걸 의심하세요.** 가끔 loss가 0.69 근처에 머무는 실행이 나올 수 있습니다.
초기 가중치를 잘못 뽑아 학습이 시작부터 막힌 경우입니다. 시드를 바꾸거나 `epochs`를 늘리면
대개 풀립니다. **신경망은 초기값 운에 영향을 받는다**는 점도 여기서 함께 알아두면 좋습니다.


In [ ]:
plt.plot(losses_single, label="은닉층 없음 (실패)")
plt.plot(losses_mlp, label="은닉층 1개 (성공)")
plt.axhline(0.693, color="gray", linestyle="--", linewidth=1)
plt.annotate("0.693 = 전부 0.5로 찍을 때", (len(losses_single) * 0.45, 0.71), color="gray")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("XOR: 은닉층의 유무")
plt.legend()
plt.show()


**결과 읽는 법** — 파란 선(은닉층 없음)은 회색 점선(0.693)에 딱 붙어 평평합니다.
주황 선(은닉층 있음)은 처음엔 같이 붙어 있다가, 어느 순간부터 뚝 떨어집니다.

그 "뚝 떨어지는 지점"이 재미있습니다. 신경망은 한동안 헤매다가 쓸 만한 중간 판단을 찾는 순간
급격히 좋아지는 경우가 많습니다. **loss가 한동안 안 내려간다고 바로 포기하면 안 되는 이유**입니다.


## 4. `loss.backward()`는 무슨 일을 하나 — 역전파

넘어가기 전에 아까 그 한 줄을 짚고 갑니다.

02번에서는 기울기를 손으로 구했습니다. 층이 하나뿐이라 가능했습니다.
그런데 방금 만든 신경망은 층이 두 개입니다. **출력층의 오차가 은닉층의 가중치를
얼마나 바꿔야 하는지**는 어떻게 알까요?

답은 **연쇄 법칙(chain rule)** 입니다. 고등학교 미분에서 배우는 그것입니다.

$$\frac{\partial \text{loss}}{\partial W_{\text{은닉}}} =
  \frac{\partial \text{loss}}{\partial \text{출력}} \times
  \frac{\partial \text{출력}}{\partial \text{은닉}} \times
  \frac{\partial \text{은닉}}{\partial W_{\text{은닉}}}$$

"출력이 1만큼 변할 때 loss가 얼마나 변하나"를 구하고, 거기에 "은닉층이 1만큼 변할 때 출력이
얼마나 변하나"를 곱하는 식으로 **뒤에서 앞으로 곱해가며** 각 층의 기울기를 구합니다.
그래서 이름이 [역전파](../../../glossary.md#backpropagation)(backpropagation, 거꾸로 전파)입니다.

**`loss.backward()` 한 줄이 이 계산 전체를 자동으로 합니다.** 층을 100개 쌓아도 똑같이 한 줄입니다.
1986년에 이 방법이 알려지면서 XOR 이후 얼어붙었던 신경망 연구가 다시 살아났습니다.

기억할 것은 하나입니다. **역전파는 "곱셈의 연쇄"** 라는 점. 이게 다음 절의 복선이 됩니다.


## 5. 진짜 데이터로 — 손글씨 숫자 7만 장

XOR은 데이터가 4개뿐인 장난감이었습니다. 이제 진짜를 해봅시다.

**MNIST**는 사람이 손으로 쓴 0~9 숫자 이미지 7만 장입니다(학습용 6만, 평가용 1만).
각 이미지는 28×28 픽셀의 흑백 사진이고, 정답은 그 숫자가 무엇인지(0~9)입니다.
머신러닝 분야에서 가장 많이 쓰인 연습 데이터라 "머신러닝의 Hello World"라고도 불립니다.

`torchvision`이 알아서 내려받아 줍니다. 처음 실행하면 다운로드에 1~2분 걸립니다.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor()])   # 이미지를 0~1 사이 숫자 텐서로 변환

train_ds = datasets.MNIST(root="../../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../../data", train=False, download=True, transform=transform)

# DataLoader: 6만 장을 한 번에 넣을 수 없으니 128장씩 잘라서 공급한다
# shuffle=True: 매 epoch마다 순서를 섞는다. 같은 순서로만 보면 그 순서까지 외워버린다
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)   # 평가는 섞을 이유가 없다

print(f"학습용 {len(train_ds)}장 / 평가용 {len(test_ds)}장")
print(f"이미지 한 장의 모양: {train_ds[0][0].shape}   <- (채널 1개, 세로 28, 가로 28)")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, (img, label) in zip(axes, train_ds):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("MNIST 샘플 8장 (제목이 정답)")
plt.show()


**결과 읽는 법** — 흐릿한 손글씨 숫자들이 보입니다. 사람이 봐도 헷갈리는 글씨가 섞여 있습니다.
28×28이면 픽셀이 784개이고, 이 **784개 숫자를 입력으로 받아 0~9 중 하나를 답하는 것**이 목표입니다.

여기서 두 가지 새 도구가 필요합니다.

**① 시그모이드 대신 [ReLU](../../../glossary.md#relu)**

$$f(x) = \max(0, x)$$

음수는 0으로 만들고 양수는 그대로 두는, 이게 전부입니다. 왜 이렇게 단순한 걸 쓸까요?

4절에서 역전파가 **곱셈의 연쇄**라고 했습니다. 그런데 시그모이드의 미분값은 아무리 커도 0.25입니다.
층이 5개면 0.25를 다섯 번 곱하게 되고, 그러면 0.001도 안 됩니다.
**앞쪽 층에 도착할 무렵에는 기울기가 사실상 0이라 학습이 멈춥니다.**
이것을 [기울기 소실](../../../glossary.md#vanishing-gradient)(vanishing gradient)이라고 합니다.

ReLU는 양수 구간의 기울기가 **정확히 1**입니다. 1은 몇 번을 곱해도 1이라 깊은 신경망에서도
기울기가 살아서 전달됩니다.

**② [드롭아웃](../../../glossary.md#dropout)**

학습 중에 뉴런 일부를 **무작위로 꺼버리는** 기법입니다. 이상한 방법 같지만 효과가 좋습니다.
특정 뉴런 하나에 과하게 의존하면 그 뉴런이 꺼졌을 때 답을 못 하므로,
신경망이 **여러 경로로 판단하도록** 강제됩니다. 학습 데이터를 통째로 외우는
[과적합](../../../glossary.md#overfitting)을 줄이는 대표적인 방법입니다.

**평가할 때는 드롭아웃을 꺼야 합니다.** 실전에서까지 뉴런을 무작위로 끌 이유가 없으니까요.
그 전환을 해주는 것이 `model.train()`과 `model.eval()`입니다.


In [ ]:
class MnistMLP(nn.Module):
    def __init__(self, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                  # 28×28 이미지를 784개짜리 한 줄로 편다
            nn.Linear(28 * 28, 256),       # 784 -> 256
            nn.ReLU(),
            nn.Dropout(dropout_p),         # 학습 중 뉴런 30%를 무작위로 끈다
            nn.Linear(256, 128),           # 256 -> 128
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 10),            # 128 -> 10 (숫자 0~9의 점수)
        )

    def forward(self, x):
        return self.net(x)


model = MnistMLP().to(device)

# Adam: 파라미터마다 학습률을 자동으로 조절하는 최적화기. SGD보다 손이 덜 간다
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# CrossEntropyLoss: 03번의 소프트맥스 + 교차 엔트로피가 한 덩어리로 들어 있다.
# 그래서 모델 마지막에 Softmax 층을 따로 붙이지 않는다 (붙이면 두 번 적용되어 학습이 나빠진다)
loss_fn = nn.CrossEntropyLoss()

print(model)


**결과 읽는 법** — 층 구조가 출력됩니다. `Linear(784 -> 256) → ReLU → Dropout → ...` 순서로
숫자가 줄어들다가 마지막에 10이 됩니다. 이 10개가 "0일 점수, 1일 점수, ..., 9일 점수"입니다.

이제 학습 루프입니다. XOR 때와 뼈대가 같고, 한 가지만 다릅니다.
**6만 장을 한 번에 못 넣으니 128장씩 나눠서 넣습니다.** 그래서 루프가 이중이 됩니다.

- 바깥 루프: epoch (전체 데이터를 한 바퀴 도는 단위)
- 안쪽 루프: batch (128장씩 한 묶음)


In [ ]:
def evaluate(model, loader):
    model.eval()          # 평가 모드 — 드롭아웃이 꺼진다
    correct, total = 0, 0
    with torch.no_grad():  # 기울기 추적을 끈다. 평가에는 필요 없고 메모리만 먹는다
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)   # 점수 10개 중 가장 큰 것의 위치 = 예측한 숫자
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total


EPOCHS = 3   # 빠르게 체감하도록 3바퀴만. 늘리면 정확도가 더 오릅니다

print(f"학습 시작 전 정확도: {evaluate(model, test_loader):.4f}  <- 아무것도 모르니 10% 근처")

for epoch in range(EPOCHS):
    model.train()        # 학습 모드 — 드롭아웃이 켜진다
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_ds)
    test_acc = evaluate(model, test_loader)
    print(f"epoch {epoch + 1}/{EPOCHS}  train_loss={train_loss:.4f}  test_acc={test_acc:.4f}")


**결과 읽는 법**

- **학습 전 정확도는 0.1 근처**입니다. 10개 중 하나를 찍는 것과 같으니 당연합니다.
  이 값이 우리의 [기준선](../../../glossary.md#baseline)입니다. 여기보다 나아야 학습된 것입니다.
- 3바퀴만 돌려도 **test 정확도가 96~97% 정도**까지 올라갑니다(환경에 따라 조금씩 다릅니다).
  784개의 픽셀값만 보고 손글씨를 100번 중 96번 맞히는 것입니다.
- `train_loss`는 epoch마다 줄어들어야 정상입니다.

**이럴 때는 이걸 의심하세요.**

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 정확도가 0.1에서 안 움직인다 | `optimizer.zero_grad()` 누락, `lr` 이상 | 코드와 `lr=1e-3` 확인 |
| loss가 `nan`이 된다 | `lr`이 너무 큼 | `lr=1e-4`로 낮추기 |
| 학습은 되는데 너무 느리다 | CPU로 도는 중 | Colab에서 GPU 켜기 |

**남은 3~4%는 왜 못 맞힐까요?** 지금 모델은 28×28 이미지를 784개 숫자로 **쭉 펴서**(Flatten)
넣었습니다. 이 순간 "어느 픽셀이 어느 픽셀 옆에 있었는지"라는 정보가 사라집니다.
사람은 획의 모양을 보고 숫자를 읽는데, 이 모델은 그걸 볼 수 없습니다.
**그 문제를 푸는 것이 다음 장의 [CNN](../../../glossary.md#cnn)입니다.**


## 정리

이번 장에서 한 일

1. 로지스틱 회귀가 XOR을 못 푸는 것을 **그림과 실험으로 확인**했습니다 (loss가 0.69에서 멈춤)
2. **은닉층 하나**를 넣자 풀렸습니다 — 직선 여러 개를 긋고 조합하는 것
3. 층이 여러 개일 때 기울기를 구하는 **역전파**를 `loss.backward()` 한 줄로 처리했습니다
4. 깊은 신경망에서 기울기가 사라지는 문제를 **ReLU**로 막았습니다
5. **드롭아웃**으로 과적합을 줄였습니다
6. MNIST 7만 장으로 **96~97% 정확도**를 얻었습니다

**스스로 확인해보기**

- [ ] XOR을 직선 하나로 못 나누는 이유를 그림으로 설명할 수 있다
- [ ] loss가 0.693에서 멈췄을 때 그게 무슨 뜻인지 안다
- [ ] 시그모이드 대신 ReLU를 쓰는 이유를 "곱셈의 연쇄"로 설명할 수 있다
- [ ] `model.train()`과 `model.eval()`을 왜 구분하는지 안다
- [ ] `optimizer.zero_grad()`를 빼먹으면 안 되는 이유를 안다

## 연습 문제

1. 3절에서 은닉층의 뉴런 수를 8에서 2로 줄이면 XOR을 여전히 풀 수 있나요?
   loss 곡선과 최종 예측값을 함께 확인해보세요.
2. 5절에서 `dropout_p`를 0으로 바꿔(드롭아웃을 끄고) 다시 학습해보세요.
   `train_loss`와 `test_acc`의 관계가 어떻게 달라지나요?
3. `EPOCHS`를 10으로 늘리거나 은닉층을 하나 더 쌓아서 정확도를 98% 이상으로 올려보세요
   (원본 강의 Lec 10의 목표와 같습니다).

**해설/정답**: [04_neural_networks_solutions.ipynb](04_neural_networks_solutions.ipynb)

다음 노트북([05_cnn.ipynb](../05_cnn/05_cnn.ipynb))에서는 이미지를 **펴지 않고 2차원 그대로**
다루는 CNN을 배웁니다. 방금 잃어버린 "픽셀 사이의 위치 관계"를 되찾는 것이 핵심입니다.
